## Selección del mejor subconjunto

Para realizar la selección del mejor subconjunto, ajustamos una regresión de mínimos cuadrados separada para cada combinación posible de los p predictores. Es decir, ajustamos todos $\binom{p}{2} = p(1-p)/2$ modelos que contienen exactamente un predictor, todos los modelos que contienen exactamente dos predictores, y así sucesivamente. Luego examinamos todos los modelos resultantes, con el objetivo de identificar el mejor.
El problema de seleccionar el mejor modelo entre las $2^p$ posibilidades consideradas por la selección del mejor subconjunto no es trivial. Esto suele dividirse en dos etapas, como se describe en el siguiente algoritmo

Best Subset Selection.
   
        1. Sea $M_0$ denote el modelo nulo, el cual no contiene ningún predictor. Este modelo simplemente predice la media muestral para cada observación.
        2.  Para $k = 1, 2, . . . p$:
        
            2.1.  Ajusta todos los $\binom{p}{k}$  modelos que contienen exactamente $k$ predictores.
            2.2. Selecciona el mejor entre $\binom{p}{k}$ modelos y llámalo $M_k$. Aquí, ''mejor'' se define como el que tiene el menor RSS (suma de los cuadrados de los residuos) o, equivalentemente, el mayor $R^2$.
        3. Selecciona un solo mejor modelo entre $M_0, \ldots, M_p$ utilizando el error de predicción cruzada ($C_p$), el criterio de información de Akaike (AIC), el criterio de información bayesiano (BIC) o el coeficiente de determinación ajustado (\(R^2\) ajustado).
      

Aunque la selección del mejor subconjunto es un enfoque simple y conceptualmente atractivo, sufre de limitaciones computacionales. El número de modelos posibles que deben considerarse crece rápidamente a medida que \(p\) aumenta. En general, hay $(2^p)$ modelos que involucran subconjuntos de \(p) predictores.

## Selección Stepwise
Por razones computacionales, la selección del mejor subconjunto no se puede aplicar con \(p\) muy grande. La selección del mejor subconjunto también puede tener problemas estadísticos cuando \(p\) es grande. Cuanto mayor sea el espacio de búsqueda, mayor será la probabilidad de encontrar modelos que parezcan buenos en los datos de entrenamiento, aunque puedan carecer de poder predictivo en datos futuros. Por lo tanto, un espacio de búsqueda enorme puede llevar al sobreajuste y a una alta varianza de las estimaciones de coeficientes.

Por ambas razones, los métodos stepwise, que exploran un conjunto mucho más restringido de modelos, son alternativas atractivas a la selección del mejor

La selección forward stepwise es una alternativa computacionalmente eficiente a la selección del mejor subconjunto. Mientras que el procedimiento de selección del mejor subconjunto considera todos los $(2^p)$ posibles modelos que contienen subconjuntos de los \(p\) predictores, forward stepwise considera un conjunto mucho más pequeño de modelos. La selección forward stepwise comienza con un modelo que no contiene ningún predictor y luego agrega predictores al modelo, uno por uno, hasta que todos los predictores están en el modelo. En particular, en cada paso se agrega al modelo la variable que proporciona la mayor mejora adicional al ajuste. Más formalmente, el procedimiento de selección forward stepwise se presenta en el siguiente algoritmo .

In [6]:
!pip install ISLP

In [7]:
!pip install l0bnb

In [8]:
# Tratamiento dedatos
import pandas as pd
import numpy as np

#Graficos
import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns

#Procesado y modelado
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import statsmodels as sms
#from sklearn.datasets import load_boston

#from sklearn.datasets import fetch_california_housing
#from sklearn.datasets import fetch_openml
from statsmodels.stats.outliers_influence import variance_inflation_factor # para calcular el VIF
# configuración de matplotlib
plt.rcParams['image.cmap']="bwr"
plt.rcParams['figure.dpi']="100"
plt.rcParams['savefig.bbox']="tight"
style.use('ggplot') or plt.style.use('ggplot')

# configuración de warnings
import warnings
warnings.filterwarnings('ignore')
# ===========================================================

from matplotlib.pyplot import subplots
from statsmodels.api import OLS
import sklearn.model_selection as skm
import sklearn.linear_model as skl
from sklearn.preprocessing import StandardScaler
from ISLP import load_data
from ISLP.models import ModelSpec as MS
from functools import partial

from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from ISLP.models import \
(Stepwise ,
sklearn_selected ,
sklearn_selection_path)
from l0bnb import fit_path

La selección forward stepwise se puede aplicar incluso en el entorno de alta dimensionalidad donde \(n < p\); sin embargo, en este caso, es posible construir solo submodelos $(M_0, \ldots, M_{n-1})$, ya que cada submodelo se ajusta mediante mínimos cuadrados, lo que no dará una solución única si $(p \geq n)$.


Forward Stepwise Selection
    
        1.  Sea $(M_0)$ denote el modelo nulo, que no contiene ningún predictr.
        2.  Para $k = 0, . . . , p-1$:
        
            2.1. Considera todos los modelos \(p - k\) que aumentan los predictores en $(M_k)$ con un predictor adicional.
            2.2. Elige el mejor entre estos \(p - k\) modelos y llámalo $(M_{k+1})$. Aquí, "mejor" se define como el que tiene el menor RSS o el mayor $(R^2\$.
        
        3. Selecciona un solo mejor modelo entre $(M_0, \ldots, M_p)$ utilizando el error de predicción cruzada $(C_p)$, el criterio de información de Akaike (AIC), el criterio de información bayesiano (BIC) o el $(R^2)$ ajustado.



In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
boston = pd.read_csv("/content/drive/MyDrive/ciencicias de datos/modulo 4/Tareas/clase 2/Boston.csv")
boston= pd.DataFrame(boston)
boston.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ciencicias de datos/modulo 4/Tareas/clase 2/Boston.csv'

In [ ]:
bostos = boston.drop("Unnamed: 0", axis=1)

In [ ]:
def nCp(sigma2 , estimator , X, Y):
    "Negative Cp statistic"
    n, p = X.shape
    Yhat = estimator.predict(X)
    RSS = np.sum((Y - Yhat)**2)
    return -(RSS + 2 * p * sigma2) / n

In [ ]:
design = MS(boston.columns.drop('medv')).fit(boston)
Y = np.array(boston['medv'])
X = design.transform(boston)
sigma2 = OLS(Y,X).fit().scale

In [ ]:
neg_Cp = partial(nCp , sigma2)
print(neg_Cp)

In [ ]:
strategy = Stepwise.first_peak(design ,direction='forward',max_terms=len(design.terms))

In [ ]:
boston_MSE = sklearn_selected(OLS ,strategy)
boston_MSE.fit(boston , Y)
boston_MSE.selected_state_

In [ ]:
boston_Cp = sklearn_selected(OLS ,strategy ,scoring=neg_Cp)# usando socring=neg_Cp se obtine el modelo con menor variables
boston_Cp.fit(boston , Y)
boston_Cp.selected_state_

## Ahora construye el modelo

In [ ]:
X3 = boston[['chas','crim','dis','lstat','nox','ptratio','rad','rm','tax','zn']]
Y = boston['medv']

X3 = sm.add_constant(X3)

mod3 = sm.OLS(Y, X3).fit()

print(mod3.summary())